# 01 – Data Exploration

Load raw market data and explore basic statistics, distributions, and correlations.

**Covers:** FRED series, Yahoo Finance prices, missing-data audit, rolling correlations.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from config.settings import DEFAULT_START_DATE, DEFAULT_END_DATE, FRED_API_KEY, DATA_DIR
from src.data.fetcher import fetch_all_data

print('Settings loaded')
print(f'Date range: {DEFAULT_START_DATE} → {DEFAULT_END_DATE}')
print(f'Data directory: {DATA_DIR}')

In [ ]:
# Load data (uses cache if available)
df = fetch_all_data(
    start_date=DEFAULT_START_DATE,
    end_date=DEFAULT_END_DATE,
    api_key=FRED_API_KEY,
    cache_dir=DATA_DIR,
)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Basic statistics
print('--- Descriptive Statistics ---')
df.describe().round(2)

In [ ]:
# Missing data audit
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct}).sort_values('missing_pct', ascending=False)

In [ ]:
# Spread history
from src.visualization.plots import plot_spread_history

spread_cols = [c for c in ['hy_spread', 'ig_spread', 'bbb_spread'] if c in df.columns]
if spread_cols:
    fig = plot_spread_history(df, spread_cols=spread_cols)
    fig.show()

In [ ]:
# Correlation heatmap
from src.visualization.plots import plot_correlation_heatmap

fig = plot_correlation_heatmap(df.select_dtypes(include='number'), window=60)
fig.show()

In [ ]:
# Distribution of HY spread
if 'hy_spread' in df.columns:
    fig = px.histogram(df, x='hy_spread', nbins=80, title='HY Spread Distribution',
                       labels={'hy_spread': 'HY Spread (bps)'})
    fig.show()